# Transmission Line Design

A Step-by-Step Superconducting Quantum Chip Design Tutorial: from Theory to Simulation

Tutorial Videos: https://www.youtube.com/playlist?list=PLnK6MrIqGXsJF6XLP-1jIBBhsCS5ZQSmR

James Saslow, Shreyan Juvvadi, Hiu Yung Wong*

contact: Hiu-Yung Wong, hiuyung.wong@sjsu.edu

In [1]:
# NOTE: Only run this cell if 'qiskit_metal' package is outside the folder where chip_version_number.ipynb is located

import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) # Allows access to qiskit_metal folder outside this folder

In [2]:
%load_ext autoreload
%autoreload 2

# Importing Packages
import numpy as np
from pandas import DataFrame


import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket 
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL
from qiskit_metal.qlibrary.qubits.transmon_pocket_teeth import TransmonPocketTeeth


from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.couplers.line_tee import LineTee

from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround


from qiskit_metal.analyses.simulation.scattering_impedance import ScatteringImpedanceSim
from qiskit_metal.analyses.quantization import LOManalysis
from qiskit_metal.renderers.renderer_ansys.ansys_renderer import QAnsysRenderer

# Coplanar Waveguide

https://www.microwaves101.com/calculators/864-coplanar-waveguide-calculator

In [3]:
# Determining Coplanar Waveguide Parameters

# ==== The Following Parameters yield an impedence of 50 ohms ====

cpw_width = 10.0 /1000  #mm # Coplanar waveguide width (The width of the blue wire)
cpw_gap   = 5.806 /1000 #mm # Ground Plane Spacing (Space between edge of blue wire and ground plane)



In [4]:
# Prompting GUI

design = designs.DesignPlanar({}, True)
design.overwrite_enabled = True # Editing Enabled

# Constraining Chip Size
design.chips.main.size['size_x'] = '5mm'
design.chips.main.size['size_y'] = '5mm'

gui = MetalGUI(design)



# Variables

In [5]:
# Quantum Chip Variables (User Input)


# ============================================================
# Chip Dimensions

margin_x = 1.46 # mm
margin_y = 0.35 # mm


vertical_lp_spacing = 2.20 #mm
bottom_lp_spacing   = 1.68 #mm
top_lp_spacing      = 3.76 #mm

# ============================================================
# Transmission Line

# coupling_length = 0.2    # mm
hanger_separation = 0.85 # mm


# ============================================================
# Bottom Qubit Placement

bottom_vert_displacement = 0.59 # m   Displacement from Bottom of the chip to the bottom of JJ(s)



# ============================================================
# Bottom Qubit 1

qubit1_bottom_shift = 0.5 #


# ============================================================
# L Bracket

L_qubit_gap = 0.005 # mm   Gap between the gray spaces of the transmon pocket and the gray region of the L-Bracket



# ============================================================
# HFSS or GDS

GDS = False # Set (GDS = True) to export to GDS and (GDS = False) for HFSS analysis






# Launch Pads

In [6]:


transmission_lpL = LaunchpadWirebond(design, 'Transmission_Launch_Pad_L',
                                options = dict(pos_x = str(-2.5 + margin_x  ) + 'mm', 
                                                      pos_y = str(-vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '0',
                                                      trace_width = cpw_width,
                                                      trace_gap = cpw_gap))


transmission_lpR = LaunchpadWirebond(design, 'Transmission_Launch_Pad_R',
                                options = dict(pos_x = str(2.5 - margin_x  ) + 'mm', 
                                                      pos_y = str(-vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '180',
                                                      trace_width = cpw_width,
                                                      trace_gap = cpw_gap))



# Making Routes Between Hangers


def make_cpw(comp1, pin1, comp2, pin2, name):
    ops = Dict(hfss_wire_bonds = False, # Set 'True' for air bridges
               trace_width = cpw_width,
               trace_gap = cpw_gap,
              pin_inputs=Dict(
                 start_pin=Dict(
                     component=comp1,
                     pin= pin1),
                 end_pin=Dict(
                     component=comp2,
                     pin=pin2)))
    
    return RouteStraight(design, name, options=ops)


cpw = make_cpw('Transmission_Launch_Pad_L', 'tie', 'Transmission_Launch_Pad_R','tie', 'cpw')


gui.rebuild()
gui.autoscale()


# HFSS Rendering for Scattering Matrix

In [7]:
from qiskit_metal.analyses.simulation.scattering_impedance import (
    ScatteringImpedanceSim
)

em1 = ScatteringImpedanceSim(design, "hfss")


In [8]:
hfss = em1.renderer
hfss.start()


INFO 05:51PM [connect_project]: Connecting to Ansys Desktop API...
INFO 05:51PM [load_ansys_project]: 	Opened Ansys App
INFO 05:51PM [load_ansys_project]: 	Opened Ansys Desktop v2026.1.2
INFO 05:51PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/012738063/Documents/Ansoft/
	Project:   Project17
INFO 05:51PM [connect_design]: No active design found (or error getting active design).
INFO 05:51PM [connect]: 	 Connected to project "Project17". No design detected


True

In [9]:
hfss.activate_ansys_design(
    "my_tl",
    "drivenmodal"
)


07:39PM 19s WARNING [activate_ansys_design]: The design_name=my_tl was not in active project.  Designs in active project are: 
[].  A new design will be added to the project.  
INFO 07:39PM [connect_design]: 	Opened active design
	Design:    my_tl [Solution type: HFSS Hybrid Modal Network]
WARNING 07:39PM [connect_setup]: 	No design setup detected.
ERROR 07:39PM [connect_setup]: Original error 😭: 'NoneType' object has no attribute 'name'



Exception:  Did you provide the correct setup name?                            Failed to pull up setup. 😭

In [10]:
hfss.render_design()

In [11]:
print(list(design.components.keys()))
print(design.components["Transmission_Launch_Pad_L"].pins.keys())

['Transmission_Launch_Pad_L', 'Transmission_Launch_Pad_R', 'cpw']
dict_keys(['tie'])
